# TPUs for LLM Fine-Tuning

This notebook demonstrates how to use Tensor Processing Units (TPUs) for fine-tuning large language models. TPUs are specialized hardware accelerators designed by Google specifically for machine learning workloads, offering significant performance advantages for LLM training and fine-tuning.

## What you'll learn
- How to set up and configure TPUs in Google Colab
- Adapting your fine-tuning code to work with TPUs
- Implementing efficient data pipelines for TPU training
- Performance comparison between TPUs, GPUs, and CPUs
- Best practices for TPU-accelerated LLM fine-tuning

**Note:** This notebook requires a TPU runtime in Google Colab. Please make sure you've selected **Runtime > Change runtime type > Hardware accelerator > TPU** before running this notebook.

In [ ]:
# Check if TPU is available
import os
import pprint
import tensorflow as tf

if 'COLAB_TPU_ADDR' in os.environ:
    print(f"TPU address: {os.environ['COLAB_TPU_ADDR']}")
    
    # TPU detection and initialization
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    
    # Print TPU cores
    print(f"TPU cores: {tf.config.list_logical_devices('TPU')}")
    strategy = tf.distribute.TPUStrategy(resolver)
    print(f"TPU strategy created with {strategy.num_replicas_in_sync} replicas")
else:
    print("No TPU detected. Please change runtime to TPU in Runtime > Change runtime type.")

## Installing Required Packages

Let's install the necessary packages for LLM fine-tuning with TPUs.

In [ ]:
# Install required packages
!pip install -q transformers datasets tensorflow-text

In [ ]:
import tensorflow as tf
import numpy as np
import time
import matplotlib.pyplot as plt
from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
from transformers import DefaultDataCollator
from datasets import load_dataset

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")

## Understanding TPU Strategy

TPUs work best with a distributed training strategy. Let's explore how to use the TPUStrategy for fine-tuning LLMs.

In [ ]:
# Define TPU strategy if available, otherwise use default strategy
if 'COLAB_TPU_ADDR' in os.environ:
    # TPU strategy already defined above
    print("Using TPU strategy")
else:
    # Fall back to default strategy
    strategy = tf.distribute.get_strategy()
    print("Using default strategy")

print(f"Number of replicas: {strategy.num_replicas_in_sync}")

## Loading a Pre-trained Model for TPU Fine-tuning

When using TPUs, we need to create the model within the TPU strategy scope.

In [ ]:
# Load tokenizer (outside strategy scope)
model_name = "distilbert-base-uncased"  # A smaller model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model within strategy scope
with strategy.scope():
    model = TFAutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,  # Binary classification for this example
        from_pt=True  # Convert from PyTorch if needed
    )
    
    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
    )

print(f"Model loaded: {model_name}")
print(f"Model size: {model.count_params():,} parameters")

## Preparing Data for TPU Fine-tuning

TPUs require special data preparation to ensure efficient training. Let's set up an optimized data pipeline.

In [ ]:
# Load a dataset for fine-tuning (SST-2 sentiment analysis dataset)
dataset = load_dataset("glue", "sst2")
print(dataset)

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Function to create TF dataset optimized for TPUs
def create_tf_dataset(dataset, batch_size=32):
    # Convert to TensorFlow dataset
    data_collator = DefaultDataCollator(return_tensors="tf")
    tf_dataset = dataset.to_tf_dataset(
        columns=["input_ids", "attention_mask"],
        label_cols=["label"],
        shuffle=True if "train" in dataset.split else False,
        batch_size=batch_size,
        collate_fn=data_collator,
    )
    
    # Optimize for TPU
    options = tf.data.Options()
    options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
    tf_dataset = tf_dataset.with_options(options)
    
    # Cache, prefetch, and optimize
    tf_dataset = tf_dataset.cache()
    tf_dataset = tf_dataset.prefetch(tf.data.AUTOTUNE)
    
    return tf_dataset

# Calculate batch size based on TPU replicas
global_batch_size = 32 * strategy.num_replicas_in_sync
print(f"Global batch size: {global_batch_size}")

# Create TPU-optimized datasets
train_dataset = create_tf_dataset(tokenized_datasets["train"], batch_size=global_batch_size)
validation_dataset = create_tf_dataset(tokenized_datasets["validation"], batch_size=global_batch_size)

print(f"Training dataset: {len(tokenized_datasets['train'])} examples")
print(f"Validation dataset: {len(tokenized_datasets['validation'])} examples")

## Fine-tuning on TPUs

Now let's fine-tune our model using TPUs for acceleration.

In [ ]:
# Define callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
]

# Train the model
print("Starting fine-tuning on TPU...")
start_time = time.time()

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=3,  # Reduced for demonstration
    callbacks=callbacks
)

training_time = time.time() - start_time
print(f"Fine-tuning completed in {training_time:.2f} seconds")

## Evaluating the TPU-Fine-tuned Model

Let's evaluate our fine-tuned model on the validation set.

In [ ]:
# Evaluate the model
evaluation = model.evaluate(validation_dataset)
print(f"Validation loss: {evaluation[0]:.4f}")
print(f"Validation accuracy: {evaluation[1]:.4f}")

## Visualizing Training Progress

Let's visualize the training and validation metrics to see how our model improved during fine-tuning.

In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss During TPU Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy During TPU Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Saving the TPU-Fine-tuned Model

Let's save our fine-tuned model for later use.

In [ ]:
# Save the model
save_directory = "./tpu_fine_tuned_model"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model saved to {save_directory}")

## Inference with the TPU-Fine-tuned Model

Let's test our fine-tuned model on some example sentences.

In [ ]:
# Test the model on some examples
test_sentences = [
    "This movie was fantastic! I really enjoyed it.",
    "The plot was confusing and the acting was terrible.",
    "It was an average film, neither good nor bad.",
    "I've never seen a better example of cinematic excellence.",
    "I fell asleep halfway through the movie."
]

# Tokenize the test sentences
inputs = tokenizer(test_sentences, padding=True, truncation=True, return_tensors="tf")

# Get predictions
outputs = model(inputs)
predictions = tf.nn.softmax(outputs.logits, axis=-1)
predicted_classes = tf.argmax(predictions, axis=-1).numpy()

# Display results
for i, sentence in enumerate(test_sentences):
    sentiment = "Positive" if predicted_classes[i] == 1 else "Negative"
    confidence = predictions[i][predicted_classes[i]].numpy() * 100
    print(f"Sentence: {sentence}")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.2f}%)\n")

## TPU vs. GPU vs. CPU Performance Comparison

Let's compare the performance of TPUs, GPUs, and CPUs for LLM fine-tuning. Note that this is a simulated comparison based on typical performance characteristics, as we can't run all three hardware types simultaneously in a single Colab notebook.

In [ ]:
# Simulated performance comparison (based on typical benchmarks)
# These values represent typical training times for a small LLM fine-tuning task
hardware = ['CPU', 'Single GPU', 'TPU v2-8', 'TPU v3-8']
training_times = [3600, 600, 180, 120]  # in seconds

# Calculate speedup relative to CPU
speedups = [training_times[0] / time for time in training_times]

# Create the plot
plt.figure(figsize=(12, 6))

# Plot training times
plt.subplot(1, 2, 1)
bars = plt.bar(hardware, training_times, color=['gray', 'green', 'blue', 'purple'])
plt.title('Training Time Comparison')
plt.ylabel('Time (seconds)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add time labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 50,
             f'{height:.0f}s',
             ha='center', va='bottom')

# Plot speedups
plt.subplot(1, 2, 2)
bars = plt.bar(hardware, speedups, color=['gray', 'green', 'blue', 'purple'])
plt.title('Speedup Relative to CPU')
plt.ylabel('Speedup (×)')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add speedup labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{height:.1f}×',
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Advantages of TPUs for LLM Fine-tuning

TPUs offer several advantages for fine-tuning large language models:

1. **Specialized Architecture**: TPUs are designed specifically for matrix operations common in deep learning, making them highly efficient for transformer models.

2. **High Memory Bandwidth**: TPUs have high-bandwidth memory (HBM) that allows for faster data access compared to GPUs.

3. **Scalability**: TPU pods can scale to thousands of chips, enabling training of extremely large models.

4. **Cost-Effectiveness**: For large workloads, TPUs often provide better performance per dollar compared to GPUs.

5. **Integrated with TensorFlow**: TPUs are designed to work seamlessly with TensorFlow, making them easy to use for LLM fine-tuning.

6. **Optimized for Transformers**: The systolic array architecture of TPUs is particularly well-suited for the matrix multiplications in transformer models.

## Best Practices for TPU-Accelerated LLM Fine-Tuning

Here are some key best practices for effectively using TPUs when fine-tuning language models:

1. **Data Pipeline Optimization**:
   - Use `tf.data` pipelines with caching and prefetching
   - Ensure your batch size is a multiple of the number of TPU cores
   - Use the `AUTO` sharding policy for optimal data distribution

2. **Model Optimization**:
   - Create and compile your model within the TPU strategy scope
   - Use bfloat16 precision which is natively supported by TPUs
   - Avoid dynamic shapes in your model architecture

3. **Training Configuration**:
   - Scale your learning rate with the global batch size
   - Use gradient accumulation for effectively larger batch sizes
   - Monitor step time to ensure TPUs are fully utilized

4. **Memory Management**:
   - Distribute large models across multiple TPU cores
   - Use gradient checkpointing for very large models
   - Consider parameter-efficient fine-tuning methods like LoRA for extremely large models

## Conclusion

In this notebook, we've explored how to use TPUs for fine-tuning large language models. We've covered:

1. Setting up and configuring TPUs in Google Colab
2. Adapting fine-tuning code to work with TPUs
3. Creating efficient data pipelines for TPU training
4. Comparing performance between TPUs, GPUs, and CPUs
5. Best practices for TPU-accelerated LLM fine-tuning

TPUs offer significant performance advantages for fine-tuning large language models, especially as model sizes continue to grow. By leveraging TPUs effectively, you can fine-tune larger models faster and more cost-effectively.

## Next Steps

To continue exploring TPU-accelerated LLM fine-tuning, consider:

1. Scaling up to larger TPU configurations (v3-8, v3-32, or TPU pods)
2. Implementing parameter-efficient fine-tuning methods like LoRA on TPUs
3. Exploring TPU-specific optimizations in TensorFlow
4. Fine-tuning larger models like T5, BERT, or GPT variants on TPUs
5. Combining TPUs with distributed training frameworks for even larger models